In [1]:
#TODO make sure this renders in the github repo

# Llama 3 Tokenizer

🌟 A tokenizer takes raw text strings and outputs a 1D list of discrete integers/tokens (e.g., "The brown rabbit..." becomes [1, 45, 102, ...]).


**Useful Links:**

- [Tokenization Visualization](https://tiktokenizer.vercel.app/?model=meta-llama%2FMeta-Llama-3-8B) 
- [HuggingFace Byte-Pair Encoding tokenization explanation](https://huggingface.co/learn/llm-course/chapter6/5)

From paper: "We use a vocabulary with $\bold{128K}$ tokens. Our token vocabulary combines $\bold{100K}$ tokens from the **tiktoken tokenizer** with $\bold{28K}$ additional tokens to better support non-English languages. Compared to the Llama $2$ tokenizer, our new tokenizer improves compression rates on a sample of English data from $3.17$ to $3.94$ characters per token. This enables the model to “read” more text for the same amount of training compute. We also found that adding $\bold{28K}$ tokens from select non-English languages improved both compression ratios and downstream performance, with no impact on English tokenization."

- They took OpenAI's pre-trained tokenizer, then they got their own dataset consisting of non-English languages and code, ran a BPE training algorithm on that dataset to generate an additional 28K BPE merge rules, and finally combined the pre-trained tokenizer with the 28K tokens to get the Llama 3 tokenizer.
- Unfortunately for my scaled down model I can not use OpenAI's pre-trained tokenizer, so I will need to train a new BPE tokenizer.

**Goals:**

- [x] Use HuggingFace's Tokenizer library to create a tokenizer that will be trained on a the FineWeb-edu dataset.

In [2]:
# @i-c
%load_ext autoreload
%autoreload 2

In [3]:
import EasyJupyter
import torch
import torch.nn as nn
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.processors import TemplateProcessing

In [4]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from llama_configs import BaseConfig

In [ ]:
import json

class BPETokenizer:
    def __init__(self, cfg: BaseConfig):
        """Handles training, saving and loading a custom BPE tokenizer for my scaled down Llama model. Not used when using Meta's Llama 3 models."""
        self.cfg = cfg

    def train_tokenizer(self, dataset_stream, chunk_size=1000, max_document = 500_000) -> Tokenizer:
        """
        Train a custom BPE tokenizer on a stream of data from HuggingFace dataset.
        Args:
            dataset_stream: A stream of raw text documents. That is created from get_raw_dataset()
            chunk_size: The number of strings that is sent to the Rust backend per iteration.
            max_document: The total number of documents to get from the stream.
        """
        cfg = self.cfg
        special_tokens = cfg.special_tokens

        def batch_iterator():
            batch = []
            for doc_idx, example in enumerate(dataset_stream):
                if doc_idx >= max_document:
                    break
                # print(f"\nDocument steam text: {example['text']}\n")
                batch.append(example['text'])
                if len(batch) == chunk_size:
                    # print(f"\nBatch length: {batch} == chuck_size: {chunk_size}")
                    # print(f"\nBatch: {batch}")
                    yield batch
                    batch = []
            if batch: # yield any remaining items
                # print(f"\nYielding remaining items")
                # print(f"\nBatch: {batch}")
                yield batch

        uni_tokenizer_dir_path = cfg.MODEL_DIR / "saved_models" / "universal_tokenizer"
        uni_tokenizer_dir_path.mkdir(parents=True, exist_ok=True)

        tokenizer_filename = f"custom_BPE_tokenizer_vocab_size_{cfg.vocab_size}.json"

        # Initialize a new Byte-Pair-Encoding tokenizer model.
        tokenizer = Tokenizer(BPE(unk_token=special_tokens["unk_token"]["token"]))
        # Using Byte-Level pre-tokenization ensures that spaces and punctuation are handled correctly without losing characters.
        tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
        # Configure the BPE trainer
        trainer = BpeTrainer(
            vocab_size=cfg.vocab_size,
            special_tokens=[v["token"] for v in special_tokens.values()],
            initial_alphabet=ByteLevel.alphabet(),
        )

        # === Train the tokenizer on the raw text file
        print(f"Training the BPE tokenizer from stream | Max Document Count: {max_document} | chunk_size: {chunk_size}...\n")
        tokenizer.train_from_iterator(batch_iterator(), trainer=trainer)

        # === Set up the post-processor to automatically add the begin_of_text token to sequences
        tokenizer.post_processor = TemplateProcessing(
            single=f"{special_tokens['doc_begin_token']["token"]} $A",
            pair=f"{special_tokens['doc_begin_token']["token"]} $A $B",
            special_tokens=[
                (
                    special_tokens["doc_begin_token"]["token"],
                    tokenizer.token_to_id(special_tokens["doc_begin_token"]["token"]),
                )
            ],
        )
        # Tell the tokenizer how to decode the token ids back into text, example: 'Ġ' back to ' '
        tokenizer.decoder = ByteLevelDecoder()

        # === Save the trained tokenizer to a JSON file
        tokenizer.save(str(uni_tokenizer_dir_path / tokenizer_filename))
        print(f"Tokenizer saved to {uni_tokenizer_dir_path / tokenizer_filename}")

        tokenizer_info_path = uni_tokenizer_dir_path / f"custom_BPE_tokenizer_vocab_size_{cfg.vocab_size}_info.json"

        with open(tokenizer_info_path, "w") as f:
            config_dict = {
                "name": tokenizer_filename,
                "description": "Custom universal tokenizer that is be to be used for all my scaled down models, do not use with any of the Llama pre-trained models like Llama 3 8B",
                "vocab_size": cfg.vocab_size,
                "max_document": max_document,
                "chunk_size": chunk_size,
            }
            json.dump(config_dict, f, indent=4)

        return tokenizer

    def load_tokenizer(self, load_test_tokenizer=False) -> tuple[bool, Tokenizer]:
        """Load the tokenizer
        
        Args:
            load_test_tokenizer: If True, load the test tokenizer that is created in tokenizer.ipynb for other notebooks to use.
        """
        if not load_test_tokenizer:
            uni_tokenizer_dir_path = self.cfg.MODEL_DIR / "saved_models" / "universal_tokenizer"
            tokenizer_path = uni_tokenizer_dir_path/ f"custom_BPE_tokenizer_vocab_size_{self.cfg.vocab_size}.json"
        else:
            uni_tokenizer_dir_path = self.cfg.MODEL_DIR / "saved_models" / "universal_tokenizer"
            tokenizer_path = uni_tokenizer_dir_path/ f"custom_BPE_tokenizer_vocab_size_{10_000}.json"  

        if tokenizer_path.exists():
            print(
                f"\nLoading existing trained BPE Tokenizer with vocab size: {self.cfg.vocab_size} from: ({tokenizer_path})..."
            )
            tokenizer = Tokenizer.from_file(str(tokenizer_path))
            return True, tokenizer
        return False, None

In [6]:
# from transformers import AutoTokenizer
# #TODO when loading Llama 3 8B model use this to load its tokenizer


# class Llama3Tokenizer:
#     def __init__(self, model_id: str = "meta-llama/Llama-3.1-8B"):
#         """
#         Load the tokenizer for the Llama 3 models. Not used for my scale down model.
#         """
#         self.tokenizer = AutoTokenizer.from_pretrained(model_id)

#     def encode(self, text: str):
#         return self.tokenizer.encode(text)

### Test: BPETokenizer()

In [7]:
# @i-c
from llama_configs import Llama3_scaled_down
from model.utils.data_loader import get_raw_dataset

cfg = Llama3_scaled_down()
cfg.vocab_size = 10_000

raw_dataset_stream = get_raw_dataset()

t = BPETokenizer(cfg)
print("\n\nTesting training a tokenizer...")
trained_tokenizer = t.train_tokenizer(raw_dataset_stream, chunk_size=1000, max_document=100)

/Users/tonyavis/miniconda3/envs/how_to_build_an_llm_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Project Root: /Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM




Testing training a tokenizer...
Training the BPE tokenizer from stream | Max Document Count: 100 | chunk_size: 1000...


Document steam text: The Independent Jane
For all the love, romance and scandal in Jane Austen’s books, what they are really about is freedom and independence. Independence of thought and the freedom to choose.
Elizabeth’s refusal of Mr. Collins offer of marriage showed an independence seldom seen in heroines of the day. Her refusal of Mr. Darcy while triggered by anger showed a level of independence that left him shocked and stunned.
The freedom she exhibited in finally accepting him in direct defiance of Lady Catherine and knowing her father would disapprove was unusual even for Austen. In her last book Anne Elliot is persuaded to refuse Captain Wentworth at Lady Russel’s insistence.
Although Jane played by the rules of the day, all of her writing is infused with how she wanted life to be. She ‘screams’ her outrage at the limitations for women in Emma.
When accos

In [8]:
# @i-c
# Test: Loading A Trained Tokenizer
success, loaded_tokenizer = t.load_tokenizer(load_test_tokenizer=True)
if success:
    print("Tokenizer successfully trained!")

    print("\n\nEncoding a string to IDs")
    text_str = "The brown rabbit ate the apple."
    encoded = loaded_tokenizer.encode(text_str)
    print(f"Original string: {text_str}")
    print(f"Token IDs: {encoded.ids}")
    print(f"Tokens: {encoded.tokens}")

    print("\n\nDecoding IDs to a string")
    decoded = loaded_tokenizer.decode(encoded.ids)
    print(f"Decoded string: {decoded}")


Loading existing trained BPE Tokenizer with vocab size: 10000 from: (/Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM/model/saved_models/universal_tokenizer/custom_BPE_tokenizer_vocab_size_10000.json)...
Tokenizer successfully trained!


Encoding a string to IDs
Original string: The brown rabbit ate the apple.
Token IDs: [1, 456, 6906, 743, 69, 4358, 8321, 266, 590, 296, 17]
Tokens: ['<|begin_of_doc|>', 'The', 'Ġbrown', 'Ġra', 'b', 'bit', 'Ġate', 'Ġthe', 'Ġapp', 'le', '.']


Decoding IDs to a string
Decoded string: The brown rabbit ate the apple.
